# 1.1.4 Control Flow

if/elif/else, for/while loops, break/continue, comprehensions — applied to Titanic data and an ML training loop simulation.

In [ ]:
import urllib.request, csv
from pathlib import Path
DATA = Path('data'); DATA.mkdir(exist_ok=True)
dest = DATA / 'titanic.csv'
if not dest.exists():
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/phihung/titanic/resolve/main/train.csv', dest)
print(f'Rows: {sum(1 for _ in open(dest))-1}')

In [ ]:
import pandas as pd
df = pd.read_csv(dest)

# if/elif for grade assignment
def age_group(age):
    if   pd.isna(age):    return 'unknown'
    elif age < 18:        return 'child'
    elif age < 35:        return 'young adult'
    elif age < 60:        return 'adult'
    else:                 return 'senior'

df['age_group'] = df['Age'].apply(age_group)
df['age_group'].value_counts()

In [ ]:
# Simulated training loop with early stopping
import math, random
random.seed(42)
history = {'train': [], 'val': []}
best, patience, no_imp = float('inf'), 4, 0
for epoch in range(1, 30):
    tl = math.exp(-epoch*0.15) + random.gauss(0, 0.02)
    vl = math.exp(-epoch*0.13) + random.gauss(0, 0.03)
    history['train'].append(tl); history['val'].append(vl)
    if vl < best - 0.001:
        best, no_imp = vl, 0
    else:
        no_imp += 1
    if no_imp >= patience:
        print(f'Early stop epoch={epoch}, best_val={best:.4f}')
        break

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history['train'], label='train loss')
plt.plot(history['val'],   label='val loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Training Curve')
plt.tight_layout(); plt.show()

In [ ]:
# List comprehensions vs loops
valid_ages = [float(a) for a in df['Age'].dropna() if 0 < float(a) < 120]
print(f'Valid ages: {len(valid_ages)}, mean: {sum(valid_ages)/len(valid_ages):.1f}')

# Nested comprehension: survival matrix
print('Survival rate matrix:')
for sex in ('male', 'female'):
    row_str = f'{sex:<8}'
    for cls in (1, 2, 3):
        sub  = df[(df.Sex == sex) & (df.Pclass == cls)]
        rate = sub.Survived.mean() if len(sub) else float('nan')
        row_str += f'  Cls{cls}: {rate:.1%}'
    print(' ', row_str)

In [ ]:
# Walrus + while pattern: chunked data processing
data = list(df['Fare'])
chunk_size = 100
idx, chunks = 0, []
while (chunk := data[idx:idx+chunk_size]):
    chunks.append(sum(chunk)/len(chunk))
    idx += chunk_size
print(f'Chunk means (first 5): {[round(c,2) for c in chunks[:5]]}')